In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

In [2]:
# # ── 1. Load dataset ──────────────────────────────────────────────
# from datasets import load_dataset
# import pandas as pd
# import re

# dataset = load_dataset("tdavidson/hate_speech_offensive")
# df = dataset["train"].to_pandas()

# # ── 2. Rename kolom biar lebih readable ─────────────────────────
# df = df.rename(columns={
#     "count":                    "annotator_count",
#     "hate_speech_count":        "hate_speech_votes",
#     "offensive_language_count": "offensive_votes",
#     "neither_count":            "neither_votes",
#     "class":                    "label",
#     "tweet":                    "text",
# })

# # ── 3. Map label integer → string ───────────────────────────────
# label_map = {0: "hate_speech", 1: "offensive", 2: "neither"}
# df["label_str"] = df["label"].map(label_map)

# # ── 4. Preprocessing teks ────────────────────────────────────────
# def preprocess_tweet(text):
#     text = str(text)
#     text = re.sub(r"http\S+|www\S+", "", text)          # hapus URL
#     text = re.sub(r"@\w+", "@USER", text)                # anonimkan mention
#     text = re.sub(r"#(\w+)", r"\1", text)                # hapus simbol hashtag, simpan kata
#     text = re.sub(r"RT\s+", "", text)                    # hapus "RT"
#     text = re.sub(r"[^\w\s@'!?.,]", " ", text)          # hapus karakter aneh
#     text = re.sub(r"\s+", " ", text).strip()             # normalisasi whitespace
#     text = text.lower()
#     return text

# df["text_clean"] = df["text"].apply(preprocess_tweet)

# # ── 5. Feature tambahan (opsional tapi useful) ───────────────────
# df["text_length"]  = df["text_clean"].str.len()
# df["word_count"]   = df["text_clean"].str.split().str.len()
# df["has_mention"]  = df["text"].str.contains(r"@\w+", regex=True).astype(int)
# df["has_hashtag"]  = df["text"].str.contains(r"#\w+", regex=True).astype(int)

# # ── 6. Buang duplikat & missing values ───────────────────────────
# df = df.drop_duplicates(subset=["text_clean"])
# df = df.dropna(subset=["text_clean", "label"])
# df = df[df["text_clean"].str.len() > 5]   # filter teks terlalu pendek

# # ── 7. Simpan ke CSV ─────────────────────────────────────────────
# df.to_csv("hate_speech_clean.csv", index=False)

# print(df.shape)
# print(df["label_str"].value_counts())
# print(df.head(3))

(24502, 12)
label_str
offensive      18990
neither         4106
hate_speech     1406
Name: count, dtype: int64
   annotator_count  hate_speech_votes  offensive_votes  neither_votes  label  \
0                3                  0                0              3      2   
1                3                  0                3              0      1   
2                3                  0                3              0      1   

                                                text  label_str  \
0  !!! RT @mayasolovely: As a woman you shouldn't...    neither   
1  !!!!! RT @mleew17: boy dats cold...tyga dwn ba...  offensive   
2  !!!!!!! RT @UrKindOfBrand Dawg!!!! RT @80sbaby...  offensive   

                                          text_clean  text_length  word_count  \
0  !!! @user as a woman you shouldn't complain ab...          126          24   
1  !!!!! @user boy dats cold...tyga dwn bad for c...           78          15   
2  !!!!!!! @user dawg!!!! @user you ever fuck a b...    

In [6]:
import pandas as pd
import re
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords, wordnet

# Download resource yang dibutuhkan
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

# ── Load dataset ─────────────────────────────────────────────────
hate_speech_df = pd.read_csv('hate_speech_clean.csv')
hate_speech_df = hate_speech_df[["text_clean", "label"]]

# ── Binary mapping ────────────────────────────────────────────────
binary_label_map = {0: 1, 1: 1, 2: 0}
hate_speech_df["label"] = hate_speech_df["label"].map(binary_label_map)
hate_speech_df = hate_speech_df.rename(columns={"text_clean": "text"})

# Fokus ke yang sering muncul di tweet & relevan ke konteks hate speech
CONTRACTIONS = {
    "ain't": "are not", "aren't": "are not", "can't": "cannot",
    "couldn't": "could not", "didn't": "did not", "doesn't": "does not",
    "don't": "do not", "hadn't": "had not", "hasn't": "has not",
    "haven't": "have not", "he'd": "he would", "he'll": "he will",
    "he's": "he is", "i'd": "i would", "i'll": "i will",
    "i'm": "i am", "i've": "i have", "isn't": "is not",
    "it's": "it is", "let's": "let us", "mightn't": "might not",
    "mustn't": "must not", "shan't": "shall not", "she'd": "she would",
    "she'll": "she will", "she's": "she is", "shouldn't": "should not",
    "that's": "that is", "there's": "there is", "they'd": "they would",
    "they'll": "they will", "they're": "they are", "they've": "they have",
    "wasn't": "was not", "we'd": "we would", "we're": "we are",
    "we've": "we have", "weren't": "were not", "what's": "what is",
    "won't": "will not", "wouldn't": "would not", "you'd": "you would",
    "you'll": "you will", "you're": "you are", "you've": "you have",
    "gonna": "going to", "wanna": "want to", "gotta": "got to",
    "kinda": "kind of", "sorta": "sort of", "lemme": "let me",
    "gimme": "give me", "tryna": "trying to",
}

def expand_contractions(text):
    """Expand contractions & informal words."""
    # Pakai word boundary untuk avoid partial match
    for contraction, expansion in CONTRACTIONS.items():
        text = re.sub(r'\b' + re.escape(contraction) + r'\b', expansion, text)
    return text

# ── POS tag → WordNet tag (untuk lemmatisasi lebih akurat) ────────
def get_wordnet_pos(treebank_tag):
    """Map NLTK POS tag ke WordNet POS tag."""
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN  # default

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

NEGATION_WORDS = {"no", "not", "nor", "never", "neither", "nobody", "nothing",
                  "nowhere", "cannot"}
stop_words = stop_words - NEGATION_WORDS

def lemmatize_text(text):
    """Tokenisasi, POS tagging, lalu lemmatisasi."""
    tokens = nltk.word_tokenize(text)
    pos_tags = nltk.pos_tag(tokens)
    lemmatized = [
        lemmatizer.lemmatize(word, get_wordnet_pos(tag))
        for word, tag in pos_tags
    ]
    return " ".join(lemmatized)

def remove_stopwords(text):
    """Hapus stopwords, kecuali negasi."""
    tokens = text.split()
    return " ".join([t for t in tokens if t not in stop_words])

# ── Pipeline preprocessing utama ─────────────────────────────────
def clean_text(text):
    if pd.isna(text):
        return ""

    text = text.lower()
    text = expand_contractions(text)           # ← BARU: expand contractions
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\bamp\b', '', text)
    text = re.sub(r'\b\d{4,}\b', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = lemmatize_text(text)                # ← BARU: lemmatisasi
    text = remove_stopwords(text)              # ← BARU: hapus stopwords
    return text

hate_speech_df['clean_tweet'] = hate_speech_df['text'].apply(clean_text)

# ── Filter & dedup ────────────────────────────────────────────────
hate_speech_df = hate_speech_df[
    hate_speech_df['clean_tweet'].str.strip() != ""
]
df = hate_speech_df.drop_duplicates(subset=["clean_tweet"])

print("NaN:", df['clean_tweet'].isna().sum())
print("Empty:", (df['clean_tweet'].str.strip() == "").sum())
print(df["label"].value_counts())
print(df.head())

df.to_csv("tdavidson_hate_speech_v0_clean.csv", index=False)

[nltk_data] Downloading package wordnet to /home/sirius/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/sirius/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to /home/sirius/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /home/sirius/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/sirius/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/sirius/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


NaN: 0
Empty: 0
label
1    19704
0     4049
Name: count, dtype: int64
                                                text  label  \
0  !!! @user as a woman you shouldn't complain ab...      0   
1  !!!!! @user boy dats cold...tyga dwn bad for c...      1   
2  !!!!!!! @user dawg!!!! @user you ever fuck a b...      1   
3       !!!!!!!!! @user @user she look like a tranny      1   
4  !!!!!!!!!!!!! @user the shit you hear about me...      1   

                                         clean_tweet  
0  woman not complain clean house man always take...  
1  boy dat coldtyga dwn bad cuffin dat hoe 1st place  
2        dawg ever fuck bitch start cry confuse shit  
3                                   look like tranny  
4     shit hear might true might faker bitch tell ya  


In [7]:
df

,text,label,clean_tweet
0,!!! @user as a woman you shouldn't complain ab...,0,woman not complain clean house man always take...
1,!!!!! @user boy dats cold...tyga dwn bad for c...,1,boy dat coldtyga dwn bad cuffin dat hoe 1st place
2,!!!!!!! @user dawg!!!! @user you ever fuck a b...,1,dawg ever fuck bitch start cry confuse shit
3,!!!!!!!!! @user @user she look like a tranny,1,look like tranny
4,!!!!!!!!!!!!! @user the shit you hear about me...,1,shit hear might true might faker bitch tell ya
...,...,...,...
24497,you's a muthaf in lie 8220 @user @user @user r...,1,yous muthaf lie right tl trash mine bible scri...
24498,"you've gone and broke the wrong heart baby, an...",0,go break wrong heart baby drive redneck crazy
24499,young buck wanna eat!!.. dat nigguh like i ain...,1,young buck want eat dat nigguh like aint fucki...
24500,youu got wild bitches tellin you lies,1,youu get wild bitch tellin lie


In [8]:
X = df["clean_tweet"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print(f"Train: {len(X_train)} | Test: {len(X_test)}")

Train: 16627 | Test: 7126


In [9]:
from sklearn.linear_model import SGDClassifier

# tfidf = TfidfVectorizer(
#     ngram_range=(1, 2),
#     min_df=1,
#     max_features=10000,
#     sublinear_tf=True
    
# )


tfidf = TfidfVectorizer(
    ngram_range=(1,2),
    min_df=2,
    max_df=0.9,
    sublinear_tf=True
)

# TfidfVectorizer(
#     analyzer="char_wb",
#     ngram_range=(3,5)
# )

# models = {
#     "LR": LogisticRegression(C=1.0, max_iter=1000, class_weight="balanced"),
#     "LinearSVM": LinearSVC(C=0.5, class_weight="balanced", max_iter=2000),
#     "RF": RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42),
# }

models = {
    "LR": LogisticRegression(
        C=1.0,
        max_iter=1000,
        class_weight="balanced"
    ),

    "LinearSVM": LinearSVC(
        C=0.5,
        class_weight="balanced",
        max_iter=2000
    ),

    # 🔥 SGD SVM variant
    "SGD_SVM": SGDClassifier(
        loss="hinge",
        penalty="l2",
        alpha=1e-4,
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ),

    # 🔥 SGD Logistic variant
    "SGD_LR": SGDClassifier(
        loss="log_loss",
        penalty="l2",
        alpha=1e-4,
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ),

    "RF": RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced",
        random_state=42
    ),
}

pipelines = {
    name: Pipeline([("tfidf", tfidf), ("clf", clf)])
    for name, clf in models.items()
}

results = {}
for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    results[name] = preds
    print(f"\n=== {name} ===")
    print(classification_report(y_test, preds, target_names=["neutral", "harmful"]))


=== LR ===
              precision    recall  f1-score   support

     neutral       0.77      0.96      0.86      1215
     harmful       0.99      0.94      0.97      5911

    accuracy                           0.94      7126
   macro avg       0.88      0.95      0.91      7126
weighted avg       0.95      0.94      0.95      7126


=== LinearSVM ===
              precision    recall  f1-score   support

     neutral       0.81      0.94      0.87      1215
     harmful       0.99      0.96      0.97      5911

    accuracy                           0.95      7126
   macro avg       0.90      0.95      0.92      7126
weighted avg       0.96      0.95      0.95      7126


=== SGD_SVM ===
              precision    recall  f1-score   support

     neutral       0.79      0.97      0.87      1215
     harmful       0.99      0.95      0.97      5911

    accuracy                           0.95      7126
   macro avg       0.89      0.96      0.92      7126
weighted avg       0.96   

In [9]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import classification_report

# 🔥 TF-IDF yang lebih optimal
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.9,
    sublinear_tf=True
)

# 🔥 Base models
lr = LogisticRegression(
    C=1.0,
    max_iter=1000,
    class_weight="balanced"
)

sgd_svm = SGDClassifier(
    loss="hinge",
    penalty="l2",
    alpha=1e-4,
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)

# 🔥 Ensemble (HARD voting)
ensemble = Pipeline([
    ("tfidf", tfidf),
    ("clf", VotingClassifier(
        estimators=[
            ("lr", lr),
            ("sgd_svm", sgd_svm)
        ],
        voting="hard",
        weights=[2, 1]   # 🔥 LR lebih dominan
    ))
])

# Train
ensemble.fit(X_train, y_train)

# Predict
y_pred = ensemble.predict(X_test)

# Evaluate
print("=== Ensemble (LR + SGD_SVM) ===")
print(classification_report(y_test, preds, target_names=["neutral", "harmful"]))

=== Ensemble (LR + SGD_SVM) ===
              precision    recall  f1-score   support

     neutral       0.84      0.66      0.74      1223
     harmful       0.93      0.97      0.95      6019

    accuracy                           0.92      7242
   macro avg       0.89      0.82      0.85      7242
weighted avg       0.92      0.92      0.92      7242



In [10]:
from sentence_transformers import SentenceTransformer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler

# 🔥 Load embedding model
model_emb = SentenceTransformer('all-MiniLM-L6-v2')

# 🔥 Encode text
X_train_emb = model_emb.encode(
    X_train.tolist(),
    batch_size=32,
    show_progress_bar=True
)

X_test_emb = model_emb.encode(
    X_test.tolist(),
    batch_size=32,
    show_progress_bar=True
)

# 🔥 IMPORTANT: scaling embedding
scaler = StandardScaler()
X_train_emb = scaler.fit_transform(X_train_emb)
X_test_emb = scaler.transform(X_test_emb)

# 🔥 STRONG classifier for embedding space
clf = LinearSVC(
    class_weight="balanced",
    C=1.0
)

# Train
clf.fit(X_train_emb, y_train)

# Predict
y_pred = clf.predict(X_test_emb)

# Evaluate
print(classification_report(y_test, y_pred, target_names=["neutral", "harmful"]))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/529 [00:00<?, ?it/s]

Batches:   0%|          | 0/227 [00:00<?, ?it/s]

              precision    recall  f1-score   support

     neutral       0.68      0.93      0.78      1223
     harmful       0.98      0.91      0.95      6019

    accuracy                           0.91      7242
   macro avg       0.83      0.92      0.86      7242
weighted avg       0.93      0.91      0.92      7242



In [11]:
from sentence_transformers import SentenceTransformer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler

# 🔥 Load embedding model
model_emb = SentenceTransformer('all-mpnet-base-v2')
# 🔥 Encode text
X_train_emb = model_emb.encode(
    X_train.tolist(),
    batch_size=32,
    show_progress_bar=True
)

X_test_emb = model_emb.encode(
    X_test.tolist(),
    batch_size=32,
    show_progress_bar=True
)

# 🔥 IMPORTANT: scaling embedding
scaler = StandardScaler()
X_train_emb = scaler.fit_transform(X_train_emb)
X_test_emb = scaler.transform(X_test_emb)

# 🔥 STRONG classifier for embedding space
SGD_LR = SGDClassifier(
    loss="log_loss",
    penalty="l2",
    alpha=1e-4,
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)

# Train
SGD_LR.fit(X_train_emb, y_train)

# Predict
y_pred = SGD_LR.predict(X_test_emb)

# Evaluate
print(classification_report(y_test, y_pred, target_names=["neutral", "harmful"]))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/529 [00:00<?, ?it/s]

Batches:   0%|          | 0/227 [00:00<?, ?it/s]

              precision    recall  f1-score   support

     neutral       0.70      0.86      0.77      1223
     harmful       0.97      0.93      0.95      6019

    accuracy                           0.91      7242
   macro avg       0.84      0.89      0.86      7242
weighted avg       0.92      0.91      0.92      7242



In [12]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
import torch
from torch.optim import AdamW
from transformers import get_scheduler
from sklearn.metrics import classification_report

MODEL_NAME = "distilbert-base-uncased"
EPOCHS = 3
BATCH_SIZE = 16  # aman untuk 6GB VRAM
LR = 2e-5

# ── Dataset wrapper ───────────────────────────────────────────────
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.encodings = tokenizer(
            texts.tolist(),
            truncation=True,
            padding=True,
            max_length=max_len,
            return_tensors="pt"
        )
        self.labels = torch.tensor(labels.tolist())

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {key: val[idx] for key, val in self.encodings.items()}, self.labels[idx]

# ── Init tokenizer & model ────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# ── DataLoader ────────────────────────────────────────────────────
train_dataset = TextDataset(X_train, y_train, tokenizer)
test_dataset  = TextDataset(X_test,  y_test,  tokenizer)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE)

# ── Optimizer & scheduler ─────────────────────────────────────────
optimizer = AdamW(model.parameters(), lr=LR)
scheduler = get_scheduler(
    "linear", optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=EPOCHS * len(train_loader)
)

# ── Training loop ─────────────────────────────────────────────────
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch, labels in train_loader:
        batch  = {k: v.to(device) for k, v in batch.items()}
        labels = labels.to(device)

        outputs = model(**batch, labels=labels)
        loss = outputs.loss
        loss.backward()

        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        total_loss += loss.item()

    print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f}")

# ── Evaluation ────────────────────────────────────────────────────
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for batch, labels in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        preds = outputs.logits.argmax(dim=-1).cpu().tolist()
        all_preds.extend(preds)
        all_labels.extend(labels.tolist())

print(classification_report(all_labels, all_preds, target_names=["neutral", "harmful"]))

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1 | Loss: 0.1309
Epoch 2 | Loss: 0.0733
Epoch 3 | Loss: 0.0413
              precision    recall  f1-score   support

     neutral       0.88      0.88      0.88      1223
     harmful       0.98      0.98      0.98      6019

    accuracy                           0.96      7242
   macro avg       0.93      0.93      0.93      7242
weighted avg       0.96      0.96      0.96      7242

